# N-ATLAS Dev Kit on Google Colab: official `NCAIR1/N-ATLaS` weights

This runs the Dev Kit's OpenAI-compatible server with NCAIR's **official** N-ATLaS weights (the `transformers` backend, loaded in 4-bit to fit a free T4), then proves it answers.

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Accept the model terms at https://huggingface.co/NCAIR1/N-ATLaS (logged in)
3. Click the **key icon** in the left sidebar → **Add new secret**: name `HF_TOKEN`, value = your Hugging Face token, **Notebook access** on
4. **Runtime → Run all**. Loading takes about 10–15 minutes the first time.

> N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.

In [ ]:
!git clone -q https://github.com/Akindats1/natlas-devkit.git
%cd natlas-devkit/server
!pip install -q -r requirements-gpu.txt

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')     # from the key-icon secret
os.environ['NATLAS_BACKEND'] = 'transformers'          # official weights, not the GGUF
os.environ['NATLAS_HF_MODEL'] = 'NCAIR1/N-ATLaS'
os.environ['NATLAS_LOAD_4BIT'] = '1'                   # fits the free T4's 15 GB
os.environ['NATLAS_API_KEY'] = 'colab-demo-key'        # clients send this as a Bearer token

In [ ]:
import subprocess, time, requests

log = open('server.log', 'w')
proc = subprocess.Popen(['python', '-m', 'natlas_server'], stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while time.time() - t0 < 1800:
    if proc.poll() is not None:
        print(open('server.log').read()[-3000:])
        raise RuntimeError('Server stopped; see the log above')
    try:
        if requests.get('http://localhost:8000/health', timeout=2).ok:
            break
    except requests.ConnectionError:
        pass
    time.sleep(10)
    print(f'loading N-ATLaS... {int(time.time() - t0)}s', end='\r')
print(f'\nServer ready in {int(time.time() - t0)}s')

In [ ]:
# Integration evidence: screenshot this cell's output
import json, subprocess, time, requests

H = {'Authorization': 'Bearer ' + os.environ['NATLAS_API_KEY']}
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:        ', gpu)
print('Model:      ', os.environ['NATLAS_HF_MODEL'], '(official weights, 4-bit)')
print('Health:     ', requests.get('http://localhost:8000/health').json())

question = 'menene ake nufi da gwagwarmaya'   # the example from NCAIR's model card
t0 = time.time()
r = requests.post('http://localhost:8000/v1/chat/completions', headers=H,
                  json={'messages': [{'role': 'user', 'content': question}], 'max_tokens': 200})
print('Powered by: ', r.headers.get('X-Powered-By'))
print('Usage:      ', r.json()['usage'], f'in {time.time() - t0:.1f}s')
print('\nQ (Hausa):', question)
print('A:', r.json()['choices'][0]['message']['content'])

### Optional: share this server publicly

Paste a free ngrok token (https://dashboard.ngrok.com/get-started/your-authtoken) below and run the cell. Any OpenAI-compatible app, the SDKs and the playground can then use the printed URL with the API key above.

In [ ]:
NGROK_TOKEN = ''  # paste your ngrok token
if NGROK_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    url = ngrok.connect(8000).public_url
    print('Playground:', url)
    print('Base URL for SDKs / OpenAI clients:', url + '/v1')
else:
    print('Skipped: no ngrok token')